### Expert Question Answerer for Netflix Shows

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace all-MiniLM-L6-v2)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


A sidebar on "temperature":
Controls how diverse the output is
A temperature of 0 means that the output should be predictable
Higher temperature for more variety in answers
Some people describe temperature as being like 'creativity' but that's not quite right

It actually controls which tokens get selected during inference
temperature=0 means: always select the token with highest probability
temperature=1 usually means: a token with 10% probability should be picked 10% of the time
Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [5]:
retriever.invoke("Which Tv Show has highest rating ?")

[Document(id='9da5ae65-9af5-48e0-a532-d93f8c4e5bc2', metadata={'row': 74, 'source': 'knowledge-base/netflix-dataset.csv', 'type': 'TV Show', 'show_type': 'TV Show'}, page_content="show_id: s75\ntitle: The World's Most Amazing Vacation Rentals\ndirector: \ncast: \ncountry: \ndate_added: September 14, 2021\nrelease_year: 2021\nrating: TV-PG\nduration: 2 Seasons\nlisted_in: Reality TV\ndescription: With an eye for every budget, three travelers visit vacation rentals around the globe and share their expert tips and tricks in this reality series."),
 Document(id='811c07eb-4e68-453a-b617-b0943c66eda0', metadata={'source': 'knowledge-base/netflix-dataset.csv', 'type': 'TV Show', 'row': 2113, 'show_type': 'TV Show'}, page_content='show_id: s2114\ntitle: High Score\ndirector: \ncast: \ncountry: United States\ndate_added: August 19, 2020\nrelease_year: 2020\nrating: TV-14\nduration: 1 Season\nlisted_in: Docuseries, Science & Nature TV\ndescription: This docuseries traces the history of classic v

In [6]:
llm.invoke("Which Tv Show has highest rating ?")

AIMessage(content='As of October 2023, one of the TV shows with the highest ratings is **"Breaking Bad"**. It has received widespread critical acclaim and holds a very high rating on platforms like IMDb (9.5/10) and Rotten Tomatoes. \n\nOther highly-rated TV shows include **"Planet Earth II,"** **"Chernobyl,"** and **"The Wire,"** which also have exceptional ratings and are often considered some of the best television series of all time.\n\nPlease note that ratings can vary depending on the platform and criteria used, and new shows may have gained popularity since then.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 14, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'syste

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the Netflix Shows.
You are chatting with a user about Netflix Shows.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Which Tv Show has highest rating ?", [])

'Based on the information provided, "Explained" has the highest rating with a TV-MA rating. However, ratings like TV-MA indicate the content is suitable for mature audiences, and this rating doesn\'t necessarily reflect viewer ratings or scores. If you\'re looking for the show with the highest viewer ratings or reviews, I would need more specific data.'

In [10]:
gr.ChatInterface(answer_question).launch()

/Users/shelkea/Desktop/llm-engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
